# FRAP Data Analysis  *(template)*
## Fit individual replicates, average them, and plot

This notebook reads **FRAP recovery curves** from Excel files (one file per
sample + ion condition), fits each replicate to a monoexponential recovery,
averages the fitted parameters, and produces comparison figures and a summary
table.

---

### How to use this notebook (for non-coders)
1. Run the cells **from top to bottom** (Shift + Enter).
2. You only need to edit **one cell**: the one marked **"EDIT THIS CELL"**.
   Lines to change are marked `# <-- CHANGE THIS`.

### Excel file format expected
- One Excel file per sample + ion, named **`{sample}_WD_{ion}.xlsx`**
  (e.g. `12nt_WD_K50.xlsx`, `T1_WD_Na50.xlsx`).
- Inside each file: the **first column is time (s)**; every other column is one
  replicate's normalised fluorescence.


## Step 1 — Load the Python tools and set the plot style

Imports the libraries (pandas, NumPy, SciPy for curve fitting, Matplotlib) and
sets global figure styling (font sizes, line widths, export resolution). Just
run it. The per-sample plot colours have moved to the **EDIT THIS CELL** below.

*First time only:* if you get a `ModuleNotFoundError`, run
`pip install pandas numpy scipy matplotlib openpyxl`.

In [ ]:
import pandas as pd
import numpy as np
import os
from scipy.optimize import curve_fit
import matplotlib.pyplot as plt

plt.rcParams['font.size'] = 16
plt.rcParams['axes.linewidth'] = 2.5
plt.rcParams['axes.labelsize'] = 16
plt.rcParams['axes.titlesize'] = 20
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial']

## Step 2 — Configuration  ✏️ **EDIT THIS CELL**

The **only** cell you need to change. Set where your Excel files live, where to
save outputs, your sample/ion names, and the plot colours.

In [ ]:
# === EDIT THIS CELL ====================================================

# Folder that CONTAINS your FRAP Excel files. Use "." for the current folder.
data_dir = "PASTE_YOUR_DATA_FOLDER_HERE"               # <-- CHANGE THIS

# Folder where figures and the summary CSV will be saved (created if missing).
output_dir = "PASTE_YOUR_OUTPUT_FOLDER_HERE"           # <-- CHANGE THIS
os.makedirs(output_dir, exist_ok=True)

# Sample names and ion conditions. File names are built as {sample}_WD_{ion}.xlsx
series_names = ['12nt', 'T1', 'T2', 'T4']              # <-- CHANGE THIS
ions = ['K50', 'Na50']                                 # <-- CHANGE THIS

# Plot colour for each sample (include every name listed in series_names).
colors = {
    '12nt': '#4DB6AC',   # Pastel Teal
    'T4':   '#64B5F6',   # Pastel Blue
    'T1':   '#EF5350',   # Pastel Red
    'T2':   '#FFA726',   # Pastel Orange
}                                                      # <-- CHANGE THIS
# ========================================================================

## Step 3 — Define the recovery model

The FRAP recovery is fitted to a single-exponential rise:
**F(t) = A · (1 − e^(−t/τ))**, where **A** is the mobile fraction (plateau) and
**τ** is the characteristic recovery time. Just run it — nothing to change.

In [ ]:
def monoexponential_recovery(t, A, tau):
    """
    Monoexponential FRAP recovery function for double-normalized data
    F(t) = A * (1 - exp(-t/tau))
    
    Parameters:
    t : time
    A : mobile fraction amplitude
    tau : recovery time constant
    """
    return A * (1 - np.exp(-t / tau))

## Step 4 — Load the replicate data from Excel

Loops over every sample × ion combination, builds the expected file name, and
loads it from `data_dir`. A tick (✓) means the file was found and its shape is
printed; a cross (✗) means it was missing — check the name/location if you see
one.

In [ ]:
# Dictionary to store raw replicate data
raw_data = {}

print("Loading raw replicate data from Excel files...")
print("=" * 70)

for series in series_names:
    for ion in ions:
        filename = f"{series}_WD_{ion}.xlsx"
        filepath = os.path.join(data_dir, filename)
        try:
            df = pd.read_excel(filepath)
            key = f"{series}_{ion}"
            raw_data[key] = df
            print(f"✓ Loaded {filename} - Shape: {df.shape}")
        except FileNotFoundError:
            print(f"✗ File not found: {filename}")
        except Exception as e:
            print(f"✗ Error loading {filename}: {str(e)}")

print("=" * 70)
print(f"\nTotal files loaded: {len(raw_data)}")

## Step 5 — Fit every replicate individually

For each Excel file, fits the model to **each replicate column separately** and
records the fitted A and τ (with their uncertainties). Fitting per replicate
(rather than to a pre-averaged curve) is what lets us estimate replicate-to-
replicate error later. The fitted values are printed as a running log.

In [ ]:
# Storage for individual replicate fits
replicate_fits = {}

# Initial guess for parameters [A, tau]
initial_guess = [0.8, 10.0]

print("=" * 70)
print("FITTING INDIVIDUAL REPLICATES")
print("=" * 70)

for key, df in raw_data.items():
    print(f"\nProcessing: {key}")
    print("-" * 70)
    
    # Get time column (first column)
    time_col = df.columns[0]
    times = df[time_col].values
    
    # Create mask for positive times
    pos_mask = times >= 0
    times_pos = times[pos_mask]
    
    # Get replicate columns (all columns except time)
    replicate_cols = df.columns[1:]
    
    replicate_fits[key] = []
    
    for rep_col in replicate_cols:
        # Get this replicate's data
        y_data = df[rep_col].values[pos_mask]
        
        try:
            # Fit the replicate
            popt, pcov = curve_fit(
                monoexponential_recovery,
                times_pos,
                y_data,
                p0=initial_guess,
                maxfev=10000
            )
            
            A, tau = popt
            perr = np.sqrt(np.diag(pcov))
            A_err, tau_err = perr
            
            replicate_fits[key].append({
                'replicate': rep_col,
                'A': A,
                'A_err': A_err,
                'tau': tau,
                'tau_err': tau_err
            })
            
            print(f"  {rep_col}: A={A:.4f}, τ={tau:.4f}")
            
        except Exception as e:
            print(f"  ✗ {rep_col}: Fitting failed - {str(e)}")
    
    print(f"  Successfully fit {len(replicate_fits[key])}/{len(replicate_cols)} replicates")

print("\n" + "=" * 70)

## Step 6 — Average the fitted parameters per sample

Takes the per-replicate A and τ values and computes the **mean ± standard error
of the mean (SEM)** for each sample × ion. SEM (= SD / √n) reflects how well the
mean is pinned down by the available replicates.

In [ ]:
# Dictionary to store averaged fit results
fit_results_avg = {}

print("=" * 70)
print("AVERAGE FITTING PARAMETERS FROM REPLICATES")
print("=" * 70)

for key, fits in replicate_fits.items():
    if len(fits) > 0:
        # Extract all A and tau values from replicates
        A_values = [fit['A'] for fit in fits]
        tau_values = [fit['tau'] for fit in fits]
        
        # Calculate mean and SEM
        A_mean = np.mean(A_values)
        A_sem = np.std(A_values, ddof=1) / np.sqrt(len(A_values))
        
        tau_mean = np.mean(tau_values)
        tau_sem = np.std(tau_values, ddof=1) / np.sqrt(len(tau_values))
        
        # Store averaged results
        fit_results_avg[key] = {
            'A': A_mean,
            'A_err': A_sem,
            'tau': tau_mean,
            'tau_err': tau_sem,
            'n_replicates': len(fits)
        }
        
        print(f"\n{key}:")
        print(f"  A = {A_mean:.4f} ± {A_sem:.4f} (n={len(fits)})")
        print(f"  τ = {tau_mean:.4f} ± {tau_sem:.4f} (n={len(fits)})")

print("\n" + "=" * 70)

## Step 7 — Split the results by ion condition

Separates the averaged results into two dictionaries (one per ion, K50 and Na50)
so each can be plotted on its own below.

In [ ]:
# Separate K50 and Na50 results
fit_results_K50 = {}
fit_results_Na50 = {}

for key, result in fit_results_avg.items():
    series, ion = key.rsplit('_', 1)
    
    if ion == 'K50':
        fit_results_K50[series] = result
    elif ion == 'Na50':
        fit_results_Na50[series] = result

print("K50 samples:", list(fit_results_K50.keys()))
print("Na50 samples:", list(fit_results_Na50.keys()))

## Step 8 — Build the time axis for plotting

Reads the time column from the first loaded file and keeps the **t ≥ 0** points
(the recovery phase, after the bleach), which is the range the fitted curves are
drawn over.

In [ ]:
# Get time array from first dataset
first_key = list(raw_data.keys())[0]
first_df = raw_data[first_key]
time_col = first_df.columns[0]
times_all = first_df[time_col].values

# Create mask for positive times (for fitting curve)
positive_time_mask = times_all >= 0
times_positive = times_all[positive_time_mask]

print(f"Time points: {len(times_all)}")
print(f"Positive time points: {len(times_positive)}")
print(f"Time range: {times_all[0]:.2f} to {times_all[-1]:.2f}")

## Step 9 — Average the raw curves for display

Separately from the fits, this averages the raw replicate **curves** point-by-
point (mean ± SEM at each time) so the figures can show the underlying data with
an error band, not just the fitted line.

In [ ]:
# Calculate mean and SEM from raw replicate data
raw_means = {}
raw_sems = {}

for key, df in raw_data.items():
    time_col = df.columns[0]
    replicate_cols = df.columns[1:]
    
    # Calculate mean and SEM across replicates
    replicate_data = df[replicate_cols].values
    mean_curve = np.mean(replicate_data, axis=1)
    sem_curve = np.std(replicate_data, axis=1, ddof=1) / np.sqrt(replicate_data.shape[1])
    
    raw_means[key] = mean_curve
    raw_sems[key] = sem_curve

print("Calculated mean curves and SEMs for all samples")

## Step 10 — Plot K⁺ samples (2×2 grid of individual fits)

One panel per sample: averaged data points with a SEM band, overlaid with the
fitted recovery curve and its τ in the legend. The `plt.savefig(...)` lines are
**commented out** by default — uncomment them to export.

In [ ]:
# Create figure for K50 samples
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

samples_order = ['12nt', 'T1', 'T2', 'T4']

for idx, sample in enumerate(samples_order):
    key = f"{sample}_K50"
    
    if sample in fit_results_K50:
        ax = axes[idx]
        
        # Get average data
        mean_curve = raw_means[key]
        sem_curve = raw_sems[key]
        
        # Get averaged fit parameters
        A = fit_results_K50[sample]['A']
        tau = fit_results_K50[sample]['tau']
        A_err = fit_results_K50[sample]['A_err']
        tau_err = fit_results_K50[sample]['tau_err']
        
        # Generate fit curve using averaged parameters
        t_fit = np.linspace(times_positive[0], times_positive[-1], 200)
        y_fit = monoexponential_recovery(t_fit, A, tau)
        
        # Get color
        color = colors[sample]
        
        # Plot with fill_between for SEM
        ax.fill_between(times_all, mean_curve - sem_curve, mean_curve + sem_curve,
                       color=color, alpha=0.3, linewidth=0)
        ax.plot(times_all, mean_curve, 'o', color=color, markersize=4, 
               alpha=0.6, label='Data (mean ± SEM)')
        
        # Plot fit
        ax.plot(t_fit, y_fit, '-', color=color, linewidth=2.5, 
               label=f'Fit: $t_{{1/2}}$={tau:.1f}±{tau_err:.1f}s')
        
        # Add labels and title
        ax.set_xlabel('Time (s)', fontsize=12, fontweight='bold')
        ax.set_ylabel('Normalized Fluorescence', fontsize=12, fontweight='bold')
        ax.set_title(f'{sample} K50', fontsize=14, fontweight='bold')
        ax.legend(fontsize=9, frameon=False)
        ax.grid(True, alpha=0.3)
        ax.set_ylim(0, 1.1)

plt.tight_layout()
#plt.savefig('K50_fits_from_averaged_params.pdf', dpi=300, bbox_inches='tight', transparent=True)
#plt.savefig('K50_fits_from_averaged_params.svg', dpi=300, bbox_inches='tight', transparent=True)
plt.show()
print("✓ K50 fit visualization saved")

## Step 11 — Plot all K⁺ samples overlaid

The same K⁺ fits and data, drawn on a single set of axes for direct comparison
between samples.

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(5, 5))

samples_order = ['12nt', 'T1', 'T2', 'T4']

for sample in samples_order:
    key = f"{sample}_K50"
    
    if sample in fit_results_K50:
        # Get average data
        mean_curve = raw_means[key]
        sem_curve = raw_sems[key]
        
        # Get averaged fit parameters
        A = fit_results_K50[sample]['A']
        tau = fit_results_K50[sample]['tau']
        
        # Generate smooth fit curve
        t_fit = np.linspace(times_positive[0], times_positive[-1], 200)
        y_fit = monoexponential_recovery(t_fit, A, tau)
        
        # Get color for this sample
        color = colors[sample]
        
        # Plot fit as solid line
        ax.plot(t_fit, y_fit, '-', color=color, linewidth=2.5)
        
        # Plot data with filled area for SEM
        ax.fill_between(times_all, mean_curve - sem_curve, mean_curve + sem_curve,
                       color=color, alpha=0.2, linewidth=0)
        
        # Plot mean data points
        ax.plot(times_all, mean_curve, 'o', color=color, markersize=3, 
               alpha=0.3, label=f'{sample}')

# Add labels and formatting
ax.set_xlabel('Time (s)')
ax.set_ylabel('Normalized Fluorescence Intensity')
#ax.legend(fontsize=12, loc='best', frameon=False)
ax.grid(False)
#ax.set_ylim(0, 0.8)  # Uncomment and adjust if needed

plt.tight_layout()
#plt.savefig('00_FRAP_Curves_svg/K50_fit_withSEM.svg', dpi=300, bbox_inches='tight', transparent=True)
#plt.savefig('01_FRAP_Curves_pdf/K50_fit_withSEM.pdf', dpi=300, bbox_inches='tight', transparent=True)
plt.show()
print("✓ K50 overlay plot saved")

## Step 12 — Plot Na⁺ samples (2×2 grid of individual fits)

Same as Step 10, for the Na⁺ condition.

In [ ]:
# Create figure for Na50 samples
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for idx, sample in enumerate(samples_order):
    key = f"{sample}_Na50"
    
    if sample in fit_results_Na50:
        ax = axes[idx]
        
        # Get average data
        mean_curve = raw_means[key]
        sem_curve = raw_sems[key]
        
        # Get averaged fit parameters
        A = fit_results_Na50[sample]['A']
        tau = fit_results_Na50[sample]['tau']
        A_err = fit_results_Na50[sample]['A_err']
        tau_err = fit_results_Na50[sample]['tau_err']
        
        # Generate fit curve using averaged parameters
        t_fit = np.linspace(times_positive[0], times_positive[-1], 200)
        y_fit = monoexponential_recovery(t_fit, A, tau)
        
        # Get color
        color = colors[sample]
        
        # Plot with fill_between for SEM
        ax.fill_between(times_all, mean_curve - sem_curve, mean_curve + sem_curve,
                       color=color, alpha=0.3, linewidth=0)
        ax.plot(times_all, mean_curve, 'o', color=color, markersize=4, 
               alpha=0.6, label='Data (mean ± SEM)')
        
        # Plot fit
        ax.plot(t_fit, y_fit, '-', color=color, linewidth=2.5, 
               label=f'Fit: $t_{{1/2}}$={tau:.1f}±{tau_err:.1f}s')
        
        # Add labels and title
        ax.set_xlabel('Time (s)', fontsize=12, fontweight='bold')
        ax.set_ylabel('Normalized Fluorescence', fontsize=12, fontweight='bold')
        ax.set_title(f'{sample} Na50', fontsize=14, fontweight='bold')
        ax.legend(fontsize=9, frameon=False)
        ax.grid(True, alpha=0.3)
        ax.set_ylim(0, 1.1)

plt.tight_layout()
#plt.savefig('Na50_fits_from_averaged_params.pdf', dpi=300, bbox_inches='tight', transparent=True)
#plt.savefig('Na50_fits_from_averaged_params.svg', dpi=300, bbox_inches='tight', transparent=True)
plt.show()

## Step 13 — Plot all Na⁺ samples overlaid

Same as Step 11, for the Na⁺ condition.

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(5, 5))

samples_order = ['12nt', 'T1', 'T2', 'T4']

for sample in samples_order:
    key = f"{sample}_Na50"
    
    if sample in fit_results_Na50:
        # Get average data
        mean_curve = raw_means[key]
        sem_curve = raw_sems[key]
        
        # Get averaged fit parameters
        A = fit_results_Na50[sample]['A']
        tau = fit_results_Na50[sample]['tau']
        
        # Generate smooth fit curve
        t_fit = np.linspace(times_positive[0], times_positive[-1], 200)
        y_fit = monoexponential_recovery(t_fit, A, tau)
        
        # Get color for this sample
        color = colors[sample]
        
        # Plot fit as solid line
        ax.plot(t_fit, y_fit, '-', color=color, linewidth=2.5)
        
        # Plot data with filled area for SEM
        ax.fill_between(times_all, mean_curve - sem_curve, mean_curve + sem_curve,
                       color=color, alpha=0.2, linewidth=0)
        
        # Plot mean data points
        ax.plot(times_all, mean_curve, 'o', color=color, markersize=3, 
               alpha=0.3, label=f'{sample}')

# Add labels and formatting
ax.set_xlabel('Time (s)')
ax.set_ylabel('Normalized Fluorescence Intensity')
#ax.legend(fontsize=12, loc='best', frameon=False)
ax.grid(False)
#ax.set_ylim(0, 0.8)  # Uncomment and adjust if needed

plt.tight_layout()
#plt.savefig('00_FRAP_Curves_svg/Na50_fit_withSEM.svg', dpi=300, bbox_inches='tight', transparent=True)
#plt.savefig('01_FRAP_Curves_pdf/Na50_fit_withSEM.pdf', dpi=300, bbox_inches='tight', transparent=True)
plt.show()
print("✓ Na50 overlay plot saved")

## Step 14 — Bar plots: mobile fraction & τ (K⁺ vs Na⁺)

Side-by-side bars comparing the two ions for each sample: **mobile fraction (A)**
on the left and **recovery time (τ)** on the right. τ for T1 is shown as *n.a.*
(its recovery is not well described by a single τ). This cell **does** save two
figures — they are routed to your `output_dir`.

In [ ]:
# Prepare data for bar plots
samples_order = ['12nt', 'T1', 'T2', 'T4']

# Extract mobile fractions (A) - all samples
mobile_fraction_K = [fit_results_K50[s]['A'] for s in samples_order if s in fit_results_K50]
mobile_fraction_Na = [fit_results_Na50[s]['A'] for s in samples_order if s in fit_results_Na50]

mobile_fraction_K_err = [fit_results_K50[s]['A_err'] for s in samples_order if s in fit_results_K50]
mobile_fraction_Na_err = [fit_results_Na50[s]['A_err'] for s in samples_order if s in fit_results_Na50]

# Extract tau values - exclude T1 (set to None)
tau_K = [fit_results_K50[s]['tau'] if s != 'T1' else None for s in samples_order if s in fit_results_K50]
tau_Na = [fit_results_Na50[s]['tau'] if s != 'T1' else None for s in samples_order if s in fit_results_Na50]

tau_K_err = [fit_results_K50[s]['tau_err'] if s != 'T1' else None for s in samples_order if s in fit_results_K50]
tau_Na_err = [fit_results_Na50[s]['tau_err'] if s != 'T1' else None for s in samples_order if s in fit_results_Na50]

# Create figure with two subplots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))

# Bar width and positions
x = np.arange(len(samples_order))
width = 0.35

# Plot 1: Mobile Fraction (A)
bars_K_A = ax1.bar(x - width/2, mobile_fraction_K, width, 
                   yerr=mobile_fraction_K_err, capsize=5,
                   color=[colors[s] for s in samples_order], 
                   alpha=0.7, edgecolor='black', linewidth=2.5,
                   label='K50')

bars_Na_A = ax1.bar(x + width/2, mobile_fraction_Na, width, 
                    yerr=mobile_fraction_Na_err, capsize=5,
                    color=[colors[s] for s in samples_order], 
                    alpha=0.5, edgecolor='black', linewidth=2.5,
                    label='Na50', hatch='//')

ax1.set_ylabel('Mobile Fraction', fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels(samples_order)

# Create custom legend with gray color
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#808080', edgecolor='black', linewidth=1.5, alpha=0.7, label='$K^+$ 50 mM'),
                   Patch(facecolor='#808080', edgecolor='black', linewidth=1.5, alpha=0.5, hatch='//', label='$Na^+$ 50 mM')]
ax1.legend(handles=legend_elements, fontsize=12, frameon=False)
ax1.set_ylim(0, max(max(mobile_fraction_K), max(mobile_fraction_Na)) * 1.2)

# Plot 2: Half-recovery time (tau) - exclude T1
# Prepare data without None values for plotting
tau_K_plot = [v if v is not None else 0 for v in tau_K]
tau_Na_plot = [v if v is not None else 0 for v in tau_Na]
tau_K_err_plot = [v if v is not None else 0 for v in tau_K_err]
tau_Na_err_plot = [v if v is not None else 0 for v in tau_Na_err]

# Create color list with gray for T1
colors_tau = [colors[s] if s != 'T1' else '#CCCCCC' for s in samples_order]

bars_K_tau = ax2.bar(x - width/2, tau_K_plot, width, 
                     yerr=tau_K_err_plot, capsize=5,
                     color=colors_tau, 
                     alpha=0.7, edgecolor='black', linewidth=2.5,
                     label='$K^+$ 50 mM')

bars_Na_tau = ax2.bar(x + width/2, tau_Na_plot, width, 
                      yerr=tau_Na_err_plot, capsize=5,
                      color=colors_tau, 
                      alpha=0.5, edgecolor='black', linewidth=2.5,
                      label='$Na^+$ 50 mM', hatch='//')

# Add "n.a." text for T1
t1_idx = samples_order.index('T1')
ax2.text(t1_idx, max([v for v in tau_K_plot + tau_Na_plot if v > 0]) * 0.1, 
         'n.a.', ha='center', va='bottom', fontweight='bold')

ax2.set_ylabel('$\tau $ (s)', fontweight='bold')
ax2.set_xticks(x)
ax2.set_xticklabels(samples_order)
ax2.set_ylim(0, max([v for v in tau_K_plot + tau_Na_plot if v > 0]) * 1.2)



plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'Tau_MobileFraction_NavsK.pdf'), dpi=300, bbox_inches='tight', transparent=True)
plt.savefig(os.path.join(output_dir, 'Tau_MobileFraction_NavsK.svg'), dpi=300, bbox_inches='tight', transparent=True)
plt.show()
print("✓ Comparison bar plots saved")

## Step 15 — Statistics: K⁺ vs Na⁺ (printout only)

Runs a two-sample t-test of K⁺ vs Na⁺ for both A and τ, using the per-replicate
fits, and prints p-values with significance stars. This cell only **reports**
the numbers; the next cell repeats the tests and also draws the annotated
figures.

> *Note:* Step 15 and Step 16 overlap by design — Step 15 is the quick numeric
> check, Step 16 is the publication figure. You can skip straight to Step 16 if
> you only want the plots.

In [ ]:
from scipy import stats

# Prepare data for bar plots
samples_order = ['12nt', 'T1', 'T2', 'T4']

# Extract mobile fractions (A) - all samples
mobile_fraction_K = [fit_results_K50[s]['A'] for s in samples_order if s in fit_results_K50]
mobile_fraction_Na = [fit_results_Na50[s]['A'] for s in samples_order if s in fit_results_Na50]

mobile_fraction_K_err = [fit_results_K50[s]['A_err'] for s in samples_order if s in fit_results_K50]
mobile_fraction_Na_err = [fit_results_Na50[s]['A_err'] for s in samples_order if s in fit_results_Na50]

# Extract tau values - exclude T1 (set to None)
tau_K = [fit_results_K50[s]['tau'] if s != 'T1' else None for s in samples_order if s in fit_results_K50]
tau_Na = [fit_results_Na50[s]['tau'] if s != 'T1' else None for s in samples_order if s in fit_results_Na50]

tau_K_err = [fit_results_K50[s]['tau_err'] if s != 'T1' else None for s in samples_order if s in fit_results_K50]
tau_Na_err = [fit_results_Na50[s]['tau_err'] if s != 'T1' else None for s in samples_order if s in fit_results_Na50]

# Perform t-tests for each sample
print("=== Statistical Tests ===\n")

# Mobile fraction t-tests
print("Mobile Fraction (A):")
p_values_A = []
for sample in samples_order:
    key_K = f"{sample}_K50"
    key_Na = f"{sample}_Na50"
    
    if key_K in replicate_fits and key_Na in replicate_fits:
        A_K_reps = [fit['A'] for fit in replicate_fits[key_K]]
        A_Na_reps = [fit['A'] for fit in replicate_fits[key_Na]]
        
        t_stat, p_val = stats.ttest_ind(A_K_reps, A_Na_reps)
        p_values_A.append(p_val)
        
        sig = "***" if p_val < 0.001 else "**" if p_val < 0.01 else "*" if p_val < 0.05 else "ns"
        print(f"  {sample}: p={p_val:.4f} {sig}")
    else:
        p_values_A.append(None)

# Tau t-tests (excluding T1)
print("\nRecovery Time (τ):")
p_values_tau = []
for sample in samples_order:
    if sample == 'T1':
        p_values_tau.append(None)
        print(f"  {sample}: excluded")
        continue
    
    key_K = f"{sample}_K50"
    key_Na = f"{sample}_Na50"
    
    if key_K in replicate_fits and key_Na in replicate_fits:
        tau_K_reps = [fit['tau'] for fit in replicate_fits[key_K]]
        tau_Na_reps = [fit['tau'] for fit in replicate_fits[key_Na]]
        
        t_stat, p_val = stats.ttest_ind(tau_K_reps, tau_Na_reps)
        p_values_tau.append(p_val)
        
        sig = "***" if p_val < 0.001 else "**" if p_val < 0.01 else "*" if p_val < 0.05 else "ns"
        print(f"  {sample}: p={p_val:.4f} {sig}")
    else:
        p_values_tau.append(None)

print("\n" + "="*30)

## Step 16 — Statistics + annotated bar plots

Recomputes the t-tests and draws two standalone figures (mobile fraction; τ) with
significance bars added above the relevant pairs. The **mobile-fraction figure is
saved** to your `output_dir`; the τ-figure's save lines are left commented.

In [ ]:
from scipy import stats

# Prepare data for bar plots
samples_order = ['12nt', 'T1', 'T2', 'T4']

# Extract mobile fractions (A) - all samples
mobile_fraction_K = [fit_results_K50[s]['A'] for s in samples_order if s in fit_results_K50]
mobile_fraction_Na = [fit_results_Na50[s]['A'] for s in samples_order if s in fit_results_Na50]

mobile_fraction_K_err = [fit_results_K50[s]['A_err'] for s in samples_order if s in fit_results_K50]
mobile_fraction_Na_err = [fit_results_Na50[s]['A_err'] for s in samples_order if s in fit_results_Na50]

# Extract tau values - exclude T1 (set to None)
tau_K = [fit_results_K50[s]['tau'] if s != 'T1' else None for s in samples_order if s in fit_results_K50]
tau_Na = [fit_results_Na50[s]['tau'] if s != 'T1' else None for s in samples_order if s in fit_results_Na50]

tau_K_err = [fit_results_K50[s]['tau_err'] if s != 'T1' else None for s in samples_order if s in fit_results_K50]
tau_Na_err = [fit_results_Na50[s]['tau_err'] if s != 'T1' else None for s in samples_order if s in fit_results_Na50]

# Perform t-tests for each sample
print("=== Statistical Tests ===\n")

# Mobile fraction t-tests
print("Mobile Fraction (A):")
p_values_A = []
for sample in samples_order:
    key_K = f"{sample}_K50"
    key_Na = f"{sample}_Na50"
    
    if key_K in replicate_fits and key_Na in replicate_fits:
        A_K_reps = [fit['A'] for fit in replicate_fits[key_K]]
        A_Na_reps = [fit['A'] for fit in replicate_fits[key_Na]]
        
        t_stat, p_val = stats.ttest_ind(A_K_reps, A_Na_reps)
        p_values_A.append(p_val)
        
        sig = "***" if p_val < 0.001 else "**" if p_val < 0.01 else "*" if p_val < 0.05 else "ns"
        print(f"  {sample}: p={p_val:.4f} {sig}")
    else:
        p_values_A.append(None)

# Tau t-tests (excluding T1)
print("\nRecovery Time (τ):")
p_values_tau = []
for sample in samples_order:
    if sample == 'T1':
        p_values_tau.append(None)
        print(f"  {sample}: excluded")
        continue
    
    key_K = f"{sample}_K50"
    key_Na = f"{sample}_Na50"
    
    if key_K in replicate_fits and key_Na in replicate_fits:
        tau_K_reps = [fit['tau'] for fit in replicate_fits[key_K]]
        tau_Na_reps = [fit['tau'] for fit in replicate_fits[key_Na]]
        
        t_stat, p_val = stats.ttest_ind(tau_K_reps, tau_Na_reps)
        p_values_tau.append(p_val)
        
        sig = "***" if p_val < 0.001 else "**" if p_val < 0.01 else "*" if p_val < 0.05 else "ns"
        print(f"  {sample}: p={p_val:.4f} {sig}")
    else:
        p_values_tau.append(None)

print("\n" + "="*30)

# Helper function to add significance bars
def add_significance_bar(ax, x_pos, y_pos, p_value, bar_height=0.02):
    """Add significance bar and asterisks above bars"""
    if p_value is None:
        return
    
    # Determine significance level
    if p_value < 0.001:
        sig_text = "***"
    elif p_value < 0.01:
        sig_text = "**"
    elif p_value < 0.05:
        sig_text = "*"
    else:
        sig_text = "ns"
    
    # Draw horizontal line connecting the two bars
    ax.plot([x_pos - 0.17, x_pos + 0.17], [y_pos, y_pos], 
            'k-', linewidth=1.5)
    
    # Add text
    ax.text(x_pos, y_pos + bar_height*0.3, sig_text, 
            ha='center', va='bottom', fontsize=12, fontweight='bold')

# Bar width and positions
x = np.arange(len(samples_order))
width = 0.35

from matplotlib.patches import Patch

# ========================================================================
# PLOT 1: Mobile Fraction (A)
# ========================================================================

fig1, ax1 = plt.subplots(1, 1, figsize=(6, 6))

bars_K_A = ax1.bar(x - width/2, mobile_fraction_K, width, 
                   yerr=mobile_fraction_K_err, capsize=5,
                   color=[colors[s] for s in samples_order], 
                   alpha=0.7, edgecolor='black', linewidth=2,
                   label='$K^+$ 50 mM')

bars_Na_A = ax1.bar(x + width/2, mobile_fraction_Na, width, 
                    yerr=mobile_fraction_Na_err, capsize=5,
                    color=[colors[s] for s in samples_order], 
                    alpha=0.5, edgecolor='black', linewidth=2,
                    label='$Na^+$ 50 mM', hatch='//')
ax1.tick_params(axis='both', labelsize=20)
ax1.set_ylabel('Mobile Fraction')
ax1.set_xticks(x)
ax1.set_xticklabels(samples_order, fontsize = 20)
# Create custom legend with gray color
legend_elements = [Patch(facecolor='#808080', edgecolor='black', linewidth=1.5, alpha=0.7), #label='$K^+$ 50 mM'),
                   Patch(facecolor='#808080', edgecolor='black', linewidth=1.5, alpha=0.5, hatch='//')]#, label='$Na^+$ 50 mM')]
ax1.legend(handles=legend_elements, fontsize=16, frameon=False)

# Set y-limit to make room for significance bars
y_max_A = max(max(mobile_fraction_K), max(mobile_fraction_Na))
y_max_err_A = max(max(mobile_fraction_K_err), max(mobile_fraction_Na_err))
ax1.set_ylim(0, (y_max_A + y_max_err_A) * 1.35)

# Add significance bars for mobile fraction
for i, (sample, p_val) in enumerate(zip(samples_order, p_values_A)):
    if p_val is not None:
        # Position bar above the taller of the two bars
        y_pos = max(mobile_fraction_K[i] + mobile_fraction_K_err[i],
                   mobile_fraction_Na[i] + mobile_fraction_Na_err[i]) + (y_max_A * 0.05)
        add_significance_bar(ax1, x[i], y_pos, p_val, bar_height=y_max_A*0.05)

plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'Mobile_Fraction_comparison_with_stats.pdf'), dpi=300, bbox_inches='tight', transparent=True)
plt.savefig(os.path.join(output_dir, 'Mobile_Fraction_comparison_with_stats.svg'), dpi=300, bbox_inches='tight', transparent=True)
plt.show()
print("✓ Mobile fraction comparison plot saved")

# ========================================================================
# PLOT 2: Half-recovery time (tau) - exclude T1
# ========================================================================

fig2, ax2 = plt.subplots(1, 1, figsize=(6, 6))

# Prepare data without None values for plotting
tau_K_plot = [v if v is not None else 0 for v in tau_K]
tau_Na_plot = [v if v is not None else 0 for v in tau_Na]
tau_K_err_plot = [v if v is not None else 0 for v in tau_K_err]
tau_Na_err_plot = [v if v is not None else 0 for v in tau_Na_err]

# Create color list with gray for T1
colors_tau = [colors[s] if s != 'T1' else '#CCCCCC' for s in samples_order]

bars_K_tau = ax2.bar(x - width/2, tau_K_plot, width, 
                     yerr=tau_K_err_plot, capsize=5,
                     color=colors_tau, 
                     alpha=0.7, edgecolor='black', linewidth=2,
                     label='K50')

bars_Na_tau = ax2.bar(x + width/2, tau_Na_plot, width, 
                      yerr=tau_Na_err_plot, capsize=5,
                      color=colors_tau, 
                      alpha=0.5, edgecolor='black', linewidth=2,
                      label='Na50', hatch='//')

# Add "n.a." text for T1
t1_idx = samples_order.index('T1')
ax2.text(t1_idx, max([v for v in tau_K_plot + tau_Na_plot if v > 0]) * 0.1, 
         'n.a.', ha='center', va='bottom')
ax2.tick_params(axis='both', labelsize=20)
ax2.set_ylabel('$\tau $ (s)')
ax2.set_xticks(x)
ax2.set_xticklabels(samples_order)

# Set y-limit to make room for significance bars
tau_values_nonzero = [v for v in tau_K_plot + tau_Na_plot if v > 0]
tau_err_nonzero = [v for v in tau_K_err_plot + tau_Na_err_plot if v > 0]
y_max_tau = max(tau_values_nonzero) if tau_values_nonzero else 1
y_max_err_tau = max(tau_err_nonzero) if tau_err_nonzero else 0
ax2.set_ylim(0, (y_max_tau + y_max_err_tau) * 1.35)

# Add significance bars for tau (excluding T1)
for i, (sample, p_val) in enumerate(zip(samples_order, p_values_tau)):
    if sample != 'T1' and p_val is not None:
        # Position bar above the taller of the two bars
        y_pos = max(tau_K_plot[i] + tau_K_err_plot[i],
                   tau_Na_plot[i] + tau_Na_err_plot[i]) + (y_max_tau * 0.05)
        add_significance_bar(ax2, x[i], y_pos, p_val, bar_height=y_max_tau*0.05)

ax2.legend(handles=legend_elements, fontsize=16, frameon=False)

plt.tight_layout()
#plt.savefig('Recovery_Time_comparison_with_stats.pdf', dpi=300, bbox_inches='tight', transparent=True)
#plt.savefig('Recovery_Time_comparison_with_stats.svg', dpi=300, bbox_inches='tight', transparent=True)
plt.show()
print("✓ Recovery time comparison plot saved")

## Step 17 — Save the summary table

Collects the averaged fit parameters into a tidy table (one row per sample × ion)
and saves it as a CSV in your `output_dir`. Open it in Excel for reporting.

In [ ]:
# Create summary dataframe
summary_records = []

for key, result in fit_results_avg.items():
    series, ion = key.rsplit('_', 1)
    summary_records.append({
        'Sample': series,
        'Ion': ion,
        'A': result['A'],
        'A_SEM': result['A_err'],
        'tau': result['tau'],
        'tau_SEM': result['tau_err'],
        'n_replicates': result['n_replicates']
    })

df_summary = pd.DataFrame(summary_records)
df_summary = df_summary.sort_values(['Sample', 'Ion'])

print("\nSummary of averaged fit parameters:")
display(df_summary)

# Save to CSV
df_summary.to_csv(os.path.join(output_dir, 'Replicates_Fit_Summary.csv'), index=False)
print("\n✓ Results saved to: fit_results_from_averaged_replicates.csv")